# Data Preprocessing Pipeline: Unemployment Rate by Country and Year

This notebook transforms raw JSON data into a structured and normalized format.

**Input:** Raw JSON dataset  
**Processing:** Type casting, column standardization, missing value handling, validation  
**Output:** Cleaned JSON dataset prepared for database upload

In [28]:
# Importing libraries
import pandas as pd
import matplotlib.pyplot as plt
import pycountry

In [30]:
# Helper function to transform country names to ISO3 (standard for all datasets in our project)
def to_iso3(name):
    try:
        return pycountry.countries.lookup(name).alpha_3
    except LookupError:
        return None

In [4]:
df = pd.read_json('raw_data\\unemployment_raw.json') # Check the directory where raw data file is stored

In [6]:
df.head()

,country,year,unemployment_rate
0,Albania,1990,9.5
1,Albania,1991,9.1
2,Albania,1992,26.5
3,Albania,1993,22.3
4,Albania,1994,18.4


In [8]:
print(df['year'].min())
print(df['year'].max())

1990
2024


In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1960 entries, 0 to 1959
Data columns (total 3 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   country            1960 non-null   object 
 1   year               1960 non-null   int64  
 2   unemployment_rate  1619 non-null   float64
dtypes: float64(1), int64(1), object(1)
memory usage: 46.1+ KB


In [12]:
df.describe()

,year,unemployment_rate
count,1960.000000,1619.000000
mean,2007.000000,8.697715
std,10.102082,5.778160
min,1990.000000,0.000000
25%,1998.000000,5.000000
50%,2007.000000,7.300000
75%,2016.000000,10.400000
max,2024.000000,38.800000


In [14]:
# leaving only 2015-2024 years (in order to analyze only latest data)
df = df[df['year']>2014]

In [18]:
df['year'].min()

2015

In [40]:
# pivoting the table so that only unique country names are in the rows
df_wide = df.pivot(index = 'country', columns = 'year', values = 'unemployment_rate')

In [42]:
df_wide.reset_index(inplace=True)

In [46]:
df_wide.head(5)

year,country,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
0,Albania,17.2,15.4,13.6,12.3,11.5,11.6,11.5,10.8,10.7,8.5
1,Andorra,3.9,3.3,2.4,1.8,2.1,2.9,3.3,2.1,1.6,1.5
2,Armenia,18.1,17.6,17.7,13.3,12.4,12.0,10.1,7.9,8.3,13.9
3,Austria,6.1,6.5,5.9,5.2,4.8,6.0,6.2,4.8,5.1,5.2
4,Azerbaijan,5.0,5.0,5.0,4.9,4.8,7.2,6.0,5.6,5.5,5.3


In [54]:
df_wide.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 56 entries, 0 to 55
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   country       56 non-null     object 
 1   2015          53 non-null     float64
 2   2016          53 non-null     float64
 3   2017          53 non-null     float64
 4   2018          53 non-null     float64
 5   2019          53 non-null     float64
 6   2020          52 non-null     float64
 7   2021          53 non-null     float64
 8   2022          52 non-null     float64
 9   2023          52 non-null     float64
 10  2024          50 non-null     float64
 11  country_code  55 non-null     object 
dtypes: float64(10), object(2)
memory usage: 5.4+ KB


In [48]:
# adding new column with standartized country name for joining with other datasets
df_wide["country_code"] = df_wide["country"].apply(to_iso3)

In [50]:
df_wide[df_wide["country_code"].isna()]

year,country,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,country_code
50,Turkiye,10.3,10.9,10.9,10.9,13.7,13.2,12.0,10.5,9.4,8.8,None


In [62]:
# manually setting country_code for Turkiye that wasn't mapped
df_wide["country_code"] = df_wide["country_code"].fillna('TUR')

In [64]:
df_wide['country_code'] = df_wide['country_code'].astype("string")

## returning back to original dataset to add a common column for manipulations with other tables (table merges): country_name + year
in the current dataset we take fields 'country_code' and 'year' to compose it

In [74]:
# adding new column with standartized country name for joining with other datasets
df["country_code"] = df["country"].apply(to_iso3)

In [76]:
df[df["country_code"].isna()]

,country,year,unemployment_rate,country_code
1775,Turkiye,2015,10.3,None
1776,Turkiye,2016,10.9,None
1777,Turkiye,2017,10.9,None
1778,Turkiye,2018,10.9,None
1779,Turkiye,2019,13.7,None
1780,Turkiye,2020,13.2,None
1781,Turkiye,2021,12.0,None
1782,Turkiye,2022,10.5,None
1783,Turkiye,2023,9.4,None
1784,Turkiye,2024,8.8,None


In [78]:
# manually setting country_code for Turkiye that wasn't mapped
df["country_code"] = df["country_code"].fillna('TUR')

In [80]:
df['country_code'] = df['country_code'].astype("string")

In [82]:
df["country_year"] = (df["country_code"] + df["year"].astype(str))

In [84]:
df.head()

,country,year,unemployment_rate,country_code,country_year
25,Albania,2015,17.2,ALB,ALB2015
26,Albania,2016,15.4,ALB,ALB2016
27,Albania,2017,13.6,ALB,ALB2017
28,Albania,2018,12.3,ALB,ALB2018
29,Albania,2019,11.5,ALB,ALB2019


## Saving preprocessed dataset in 2 versions: wide (modified) and long (non-unique country_names)

In [88]:
# wide format (country, 2015, ..., 2024, country_code)
df_wide.to_json(
    "processed_data/unemployement_rate_wide_clean.json",
    orient="records",
    indent=2,
    force_ascii=False
)

In [86]:
# original format with сountry_name + year as a column for further manipulations with other datasets
df.to_json(
    "processed_data/unemployement_rate_clean.json",
    orient="records",
    indent=2,
    force_ascii=False
)